# 01. SOVIGAZ DATA AUDIT & EXPLORATORY PROFILING (PHASE 1)
**Project**: SOVIGAZ Machine Learning System  
**Author**: Senior Machine Learning Engineer & Data Scientist  
**Purpose**: Khảo sát toàn diện 19 bảng dữ liệu DWH Sovigaz, kiểm tra tính toàn vẹn hình sao (Star Schema), chống rò rỉ dữ liệu (Data Leakage), và xác lập cơ sở thực nghiệm cho 3 bài toán Machine Learning.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
sys.path.insert(0, str(project_root))

from config.config import RAW_DATA_DIR, RANDOM_STATE
print(f"Project Root: {project_root}")
print(f"Raw Data Dir: {RAW_DATA_DIR}")


## 1. Kiểm kê danh mục toàn bộ các bảng trong DWH

In [ ]:
import glob

csv_files = sorted(glob.glob(str(RAW_DATA_DIR / "*.csv")))
summary_list = []
for f in csv_files:
    tname = Path(f).stem
    size_mb = os.path.getsize(f) / (1024 * 1024)
    df_temp = pd.read_csv(f, low_memory=False)
    summary_list.append({
        'Table': tname,
        'File Size (MB)': round(size_mb, 2),
        'Rows': len(df_temp),
        'Cols': df_temp.shape[1],
        'Duplicate Rows': df_temp.duplicated().sum()
    })

df_summary = pd.DataFrame(summary_list)
display(df_summary)


## 2. Kiểm tra tính toàn vẹn liên kết Star Schema & Join Explosion

In [ ]:
# Đọc Fact chính
rev = pd.read_csv(RAW_DATA_DIR / "DM_RevenueBySaleLedger.csv", usecols=['RefNo', 'RefDate', 'Revenue', 'AccountObjectCode', 'InventoryItemCode_New'], low_memory=False)
cust = pd.read_csv(RAW_DATA_DIR / "Dim_Customer.csv", usecols=['AccountObjectCode', 'CustomerName', 'StockCode'], low_memory=False).drop_duplicates(subset=['AccountObjectCode'])
prod = pd.read_csv(RAW_DATA_DIR / "Dim_Products.csv", usecols=['InventoryItemCode_New', 'Excel_Category_Name'], low_memory=False).drop_duplicates(subset=['InventoryItemCode_New'])

before_len = len(rev)
m1 = rev.merge(cust, on='AccountObjectCode', how='left')
m2 = m1.merge(prod, on='InventoryItemCode_New', how='left')
after_len = len(m2)

print(f"Số dòng Fact ban đầu : {before_len:,}")
print(f"Số dòng sau khi Join : {after_len:,}")
print(f"Tỷ lệ khớp khách hàng: {(m2['CustomerName'].notna().mean()*100):.2f}%")
print(f"Tỷ lệ khớp sản phẩm  : {(m2['Excel_Category_Name'].notna().mean()*100):.2f}%")
assert before_len == after_len, "LỖI: Bùng nổ số dòng sau khi Join!"
print("=> KẾT QUẢ: Toàn vẹn 100%, không bị Join Explosion!")


## 3. Khảo sát chu kỳ mua hàng B2B & Cơ sở xác định ngưỡng Churn

In [ ]:
rev['RefDate'] = pd.to_datetime(rev['RefDate'])
cust_orders = rev[['AccountObjectCode', 'RefDate']].drop_duplicates().sort_values(by=['AccountObjectCode', 'RefDate'])
cust_orders['Prev_Date'] = cust_orders.groupby('AccountObjectCode')['RefDate'].shift(1)
cust_orders['Interval'] = (cust_orders['RefDate'] - cust_orders['Prev_Date']).dt.days

intervals = cust_orders['Interval'].dropna()

print(f"Trung vị (Median) : {intervals.quantile(0.50):.0f} ngày")
print(f"Phân vị 75 (P75)  : {intervals.quantile(0.75):.0f} ngày")
print(f"Phân vị 90 (P90)  : {intervals.quantile(0.90):.0f} ngày")
print(f"Tỷ lệ mua lại <= 60 ngày: {(intervals <= 60).mean()*100:.2f}%")
print(f"Tỷ lệ mua lại <= 90 ngày: {(intervals <= 90).mean()*100:.2f}%")

plt.figure(figsize=(10, 4))
sns.histplot(intervals[intervals <= 120], bins=60, kde=True, color='#1F497D')
plt.axvline(intervals.quantile(0.50), color='green', linestyle='--', label=f'Median: {intervals.quantile(0.50):.0f}d')
plt.axvline(60, color='orange', linestyle='--', label='High Risk Threshold (60d - 98.2%)')
plt.axvline(90, color='red', linestyle='--', label='Definitive Churn (90d - 99.0%)')
plt.title("Phân phối khoảng cách giữa các lần mua hàng lặp lại (B2B Purchase Interval)")
plt.xlabel("Số ngày giữa hai đơn hàng liên tiếp")
plt.ylabel("Số lượt phát sinh")
plt.legend()
plt.tight_layout()
plt.show()
